## 한국어 뉴스 주제 분류 파인튜닝

기본 모델: `klue/bert-base`  
데이터셋: `klue/ynat`  
결과: 한국어 뉴스 제목을 주제별로 분류

파인튜닝 전에는 기본 모델이 뉴스 주제 분류용으로 학습되어 있지 않기 때문에 원하는 라벨을 제대로 예측하지 못한다.  
파인튜닝 후에는 한국어 뉴스 제목을 입력했을 때 정치, 경제, 사회, 생활문화, 세계, IT과학, 스포츠 중 하나로 분류할 수 있다.

영화리뷰 감정분석
기본 모델: beomi/kcbert-base
데이터셋: NSMC
분류 라벨: 부정, 긍정
결과: 한국어 영화 리뷰의 감정을 분류

In [ ]:
!pip install -q -U "datasets<4.0.0" transformers evaluate accelerate

In [ ]:
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
import evaluate

print("GPU 사용 가능 여부:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))

데이터 불러오기

In [ ]:
from datasets import load_dataset

# 1. 최신 허깅페이스 정책에 맞춰 데이터셋 주소를 'klue/klue'로 정확하게 명시합니다.
ds = load_dataset("klue/klue", "ynat")

# 데이터 구조 확인 (train, validation 데이터가 잘 나오는지 확인하세요)
print(ds)

기본 모델과 토크나이저 불러오기

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "klue/bert-base"

# 1. 토크나이저 불러오기
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. 분류용 모델 불러오기 (뉴스 주제 라벨 개수에 맞춰 num_labels 설정. 예: 7개)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=7)

In [ ]:
# 1. 뉴스 제목('title')을 모델 규격에 맞게 변환하는 함수 정의
def tokenize_function(examples):
    return tokenizer(examples["title"], padding="max_length", truncation=True, max_length=64)

# 2. 전체 데이터셋(학습용, 검증용)에 일괄 적용
tokenized_datasets = ds.map(tokenize_function, batched=True)

print("데이터 전처리가 완료되었습니다!")

In [ ]:
import numpy as np
import evaluate
from transformers import TrainingArguments, Trainer

# 1. 평가 기준(정확도) 설정
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 2. 학습 하이퍼파라미터 설정
training_args = TrainingArguments(
    output_dir="./news_topic_model",
    eval_strategy="epoch",      # 1에포크(전체 데이터 1회 학습)마다 시험 보기
    learning_rate=2e-5,          # 미세조정(Fine-tuning)을 위한 안정적인 학습률
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,          # 전체 데이터를 총 3번 반복해서 공부
    weight_decay=0.01,
    logging_steps=100,           # 100걸음마다 로그 출력
)

# 3. 학습을 담당할 Trainer 객체 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

# 4. 🔥 본격적인 학습 시작! (시간이 좀 걸리니 완료될 때까지 기다려주세요)
trainer.train()

In [ ]:
# 1. 모델의 최종 정확도(성적표) 확인하기
print("--- [모델 최종 성적표] ---")
results = trainer.evaluate()
accuracy_percent = results['eval_accuracy'] * 100
print(f"최종 정확도: {accuracy_percent:.2f}%")
print("-" * 25 + "\n")

# 2. 테스트하고 싶은 뉴스 제목 입력
test_text = "정부, 내달부터 소상공인 금융지원 대폭 확대 결정"

# 3. 모델이 이해할 수 있는 형태로 변환 및 예측
inputs = tokenizer(test_text, return_tensors="pt", truncation=True, max_length=64)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    predicted_class_id = logits.argmax().item()

# 4. 결과 출력
labels = ["IT과학", "경제", "사회", "생활문화", "세계", "스포츠", "정치"]

print(f"입력 뉴스: '{test_text}'")
print(f"👉 AI가 분석한 주제: [{labels[predicted_class_id]}]")

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. NSMC(네이버 영화 리뷰) 데이터셋 불러오기
nsmc_ds = load_dataset("e9t/nsmc")

# 2. 감정 분석 기본 모델 지정
sentiment_model_name = "beomi/kcbert-base"

# 3. 토크나이저 및 모델 불러오기 (긍정/부정 2개 라벨이므로 num_labels=2)
sentiment_tokenizer = AutoTokenizer.from_pretrained(sentiment_model_name)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(sentiment_model_name, num_labels=2)

# 4. 영화 리뷰 텍스트('document' 컬럼) 토큰화 함수
def tokenize_nsmc(examples):
    # NSMC 데이터에는 빈 텍스트(None)가 섞여 있을 수 있어 문자로 확실히 변환해줍니다.
    texts = [str(doc) if doc is not None else "" for doc in examples["document"]]
    return sentiment_tokenizer(texts, padding="max_length", truncation=True, max_length=64)

# 5. 전체 리뷰 데이터 토큰화 적용
tokenized_nsmc = nsmc_ds.map(tokenize_nsmc, batched=True)

print("영화 리뷰 데이터 및 모델 세팅 완료!")

In [ ]:
from transformers import TrainingArguments, Trainer
import numpy as np
import evaluate

# 1. 평가 지표 로드
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 2. 학습 설정
sentiment_training_args = TrainingArguments(
    output_dir="./sentiment_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=500,
)

# 3. [수정됨] 데이터를 2만 개(train)와 2천 개(test)로 줄여서 빠르게 학습 설정
small_train_dataset = tokenized_nsmc["train"].shuffle(seed=42).select(range(20000))
small_eval_dataset = tokenized_nsmc["test"].shuffle(seed=42).select(range(2000))

# 4. 영화 리뷰용 트레이너 객체 (축소된 데이터셋 적용)
sentiment_trainer = Trainer(
    model=sentiment_model,
    args=sentiment_training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)

# 5. 🔥 감정 분석 파인튜닝 학습 시작!
sentiment_trainer.train()

In [ ]:
# 1. 영화 리뷰 모델의 최종 성적표 확인하기
print("--- [영화 리뷰 모델 성적표] ---")
sentiment_results = sentiment_trainer.evaluate()
accuracy_percent = sentiment_results['eval_accuracy'] * 100
print(f"최종 정확도: {accuracy_percent:.2f}%")
print("-" * 28 + "\n")

# 2. 테스트하고 싶은 영화 리뷰 입력
review_text = "너무 재미가없네요."

# 3. 모델이 이해할 수 있는 형태로 변환 및 예측
inputs = sentiment_tokenizer(review_text, return_tensors="pt", truncation=True, max_length=64)
inputs = {k: v.to(sentiment_model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = sentiment_model(**inputs)
    logits = outputs.logits
    predicted_class_id = logits.argmax().item()

# 4. 결과 출력 (0: 부정, 1: 긍정)
labels = ["부정 👎", "긍정 👍"]

print(f"입력 리뷰: '{review_text}'")
print(f"👉 AI 감정 분석 결과: [{labels[predicted_class_id]}]")